In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report
import timm
import kagglehub

MODEL_NAME = 'efficientnet_b0'
EPOCHS = 10
BATCH_SIZE = 64

In [3]:
dataset_path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
csv_path = os.path.join(dataset_path, 'HAM10000_metadata.csv')

class HAM10000Dataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_name = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        if not os.path.exists(img_name):
            for root, dirs, files in os.walk(self.img_dir):
                if f"{img_id}.jpg" in files:
                    img_name = os.path.join(root, f"{img_id}.jpg")
                    break

        image = Image.open(img_name).convert('RGB')
        label = self.class_to_idx[self.df.iloc[idx]['dx']]
        
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = HAM10000Dataset(csv_path, dataset_path, transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_subset, test_subset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cnn_model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=7)

if torch.cuda.device_count() > 1:
    cnn_model = nn.DataParallel(cnn_model)

cnn_model = cnn_model.to(device)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    cnn_model.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

100%|██████████| 126/126 [02:53<00:00,  1.38s/it]


Epoch 1 Loss: 1.2658


100%|██████████| 126/126 [02:21<00:00,  1.13s/it]


Epoch 2 Loss: 0.4322


100%|██████████| 126/126 [02:22<00:00,  1.13s/it]


...

Epoch 8 Loss: 0.0345


100%|██████████| 126/126 [02:26<00:00,  1.16s/it]


Epoch 9 Loss: 0.0272


100%|██████████| 126/126 [04:11<00:00,  1.99s/it]

Epoch 10 Loss: 0.0308


In [8]:
cnn_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        outputs = cnn_model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

save_path = '/kaggle/working/efficientnet_b0_baseline.pth'
if isinstance(cnn_model, nn.DataParallel):
    torch.save(cnn_model.module.state_dict(), save_path)
else:
    torch.save(cnn_model.state_dict(), save_path)

100%|██████████| 32/32 [00:57<00:00,  1.78s/it]

              precision    recall  f1-score   support

       akiec       0.58      0.56      0.57        75
         bcc       0.76      0.65      0.70       105
         bkl       0.74      0.53      0.62       202
          df       0.67      0.37      0.48        27
         mel       0.57      0.55      0.56       228
          nv       0.89      0.95      0.92      1339
        vasc       0.75      0.78      0.76        27

    accuracy                           0.82      2003
   macro avg       0.71      0.63      0.66      2003
weighted avg       0.81      0.82      0.81      2003



In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vit_model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=7)

if torch.cuda.device_count() > 1:
    vit_model = nn.DataParallel(vit_model)

vit_model = vit_model.to(device)

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

In [10]:
EPOCHS = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vit_model.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    vit_model.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = vit_model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

100%|██████████| 126/126 [03:39<00:00,  1.74s/it]


Epoch 1 Loss: 0.6613


100%|██████████| 126/126 [02:51<00:00,  1.36s/it]


Epoch 2 Loss: 0.3527


100%|██████████| 126/126 [02:41<00:00,  1.28s/it]


...

Epoch 8 Loss: 0.0366


100%|██████████| 126/126 [02:35<00:00,  1.24s/it]


Epoch 9 Loss: 0.0152


100%|██████████| 126/126 [02:45<00:00,  1.31s/it]

Epoch 10 Loss: 0.0728


In [11]:
vit_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        outputs = vit_model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

save_path = '/kaggle/working/vit_small_baseline.pth'
if isinstance(vit_model, nn.DataParallel):
    torch.save(vit_model.module.state_dict(), save_path)
else:
    torch.save(vit_model.state_dict(), save_path)

100%|██████████| 32/32 [00:46<00:00,  1.45s/it]


              precision    recall  f1-score   support

       akiec       0.66      0.49      0.56        75
         bcc       0.90      0.66      0.76       105
         bkl       0.89      0.50      0.64       202
          df       0.78      0.67      0.72        27
         mel       0.74      0.50      0.60       228
          nv       0.86      0.99      0.92      1339
        vasc       0.89      0.93      0.91        27

    accuracy                           0.85      2003
   macro avg       0.82      0.68      0.73      2003
weighted avg       0.84      0.85      0.83      2003

